# Backtest Visualization

Run the EMA Cross + TWAP backtest and visualize results.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from decimal import Decimal
from nautilus_trader.adapters.binance import BINANCE_VENUE
from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.examples.algorithms.twap import TWAPExecAlgorithm
from nautilus_trader.examples.strategies.ema_cross_twap import EMACrossTWAP, EMACrossTWAPConfig
from nautilus_trader.model.data import BarType
from nautilus_trader.model.enums import AccountType, BookType, OmsType
from nautilus_trader.model.identifiers import TraderId
from nautilus_trader.model.currencies import ETH, USDT
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import TradeTickDataWrangler
from nautilus_trader.test_kit.providers import TestDataProvider, TestInstrumentProvider

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 7)
print('Libraries loaded.')

In [ ]:
# Run backtest
config = BacktestEngineConfig(
    trader_id=TraderId("BACKTESTER-001"),
    logging=LoggingConfig(log_level="WARN", use_pyo3=False),
)

engine = BacktestEngine(config=config)
engine.add_venue(
    venue=BINANCE_VENUE,
    oms_type=OmsType.NETTING,
    book_type=BookType.L1_MBP,
    account_type=AccountType.CASH,
    base_currency=None,
    starting_balances=[Money(1_000_000.0, USDT), Money(10.0, ETH)],
)

ETHUSDT_BINANCE = TestInstrumentProvider.ethusdt_binance()
engine.add_instrument(ETHUSDT_BINANCE)

provider = TestDataProvider()
wrangler = TradeTickDataWrangler(instrument=ETHUSDT_BINANCE)
ticks = wrangler.process(provider.read_csv_ticks("binance/ethusdt-trades.csv"))
engine.add_data(ticks)
print(f'Loaded {len(ticks)} trade ticks')

strategy_config = EMACrossTWAPConfig(
    instrument_id=ETHUSDT_BINANCE.id,
    bar_type=BarType.from_str("ETHUSDT.BINANCE-250-TICK-LAST-INTERNAL"),
    trade_size=Decimal("0.10"),
    fast_ema_period=10,
    slow_ema_period=20,
    twap_horizon_secs=10.0,
    twap_interval_secs=2.5,
)
strategy = EMACrossTWAP(config=strategy_config)
engine.add_strategy(strategy=strategy)
engine.add_exec_algorithm(TWAPExecAlgorithm())

engine.run()
print('Backtest complete.')

In [ ]:
# === 1. Account Equity Curve ===
account_report = engine.trader.generate_account_report(BINANCE_VENUE)
usdt_bal = account_report[account_report['currency'] == 'USDT'].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(usdt_bal.index, usdt_bal['total'], label='USDT Balance', color='#2196F3', linewidth=1.5)
ax.fill_between(usdt_bal.index, usdt_bal['total'], alpha=0.15, color='#2196F3')
ax.set_title('Account Equity Curve (USDT)', fontsize=14, fontweight='bold')
ax.set_ylabel('Balance (USDT)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.tight_layout()
plt.show()

usdt_start = float(usdt_bal.iloc[0]['total'])
usdt_end = float(usdt_bal.iloc[-1]['total'])
print(f'Start: {usdt_start:,.2f} USDT  ->  End: {usdt_end:,.2f} USDT  (PnL: {usdt_end - usdt_start:+.2f} USDT)')

In [ ]:
# === 2. Positions PnL Bar Chart ===
positions_report = engine.trader.generate_positions_report().reset_index()

def money_to_float(v):
    if hasattr(v, 'as_double'):
        return float(v.as_double())
    return float(str(v).split()[0])

positions_report['pnl'] = positions_report['realized_pnl'].apply(money_to_float)
positions_report['side_label'] = positions_report['entry'].apply(lambda x: 'LONG' if str(x).strip() == 'BUY' else 'SHORT')

colors = ['#4CAF50' if p > 0 else '#F44336' for p in positions_report['pnl']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PnL per trade
bars = ax1.bar(range(len(positions_report)), positions_report['pnl'], color=colors, edgecolor='white')
ax1.axhline(0, color='black', linewidth=0.5)
ax1.set_title('PnL per Trade', fontsize=13, fontweight='bold')
ax1.set_xlabel('Trade #')
ax1.set_ylabel('PnL (USDT)')
ax1.grid(True, alpha=0.3, axis='y')

# Cumulative PnL
cumsum = positions_report['pnl'].cumsum()
ax2.plot(range(len(cumsum)), cumsum, color='#FF9800', linewidth=2, marker='o', markersize=4)
ax2.fill_between(range(len(cumsum)), cumsum, alpha=0.15, color='#FF9800')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_title('Cumulative PnL', fontsize=13, fontweight='bold')
ax2.set_xlabel('Trade #')
ax2.set_ylabel('Cumulative PnL (USDT)')
ax2.grid(True, alpha=0.3)

wins = len(positions_report[positions_report['pnl'] > 0])
total = len(positions_report)
print(f'Win Rate: {wins}/{total} ({wins/total*100:.1f}%)')
print(f'Total PnL: {positions_report["pnl"].sum():+.2f} USDT')
plt.tight_layout()
plt.show()

In [ ]:
# === 3. Price Chart with Entry/Exit Markers ===
# Build OHLC from trade ticks
ticks = wrangler.process(provider.read_csv_ticks("binance/ethusdt-trades.csv"))

# Convert TradeTick list to a DataFrame
tick_df = pd.DataFrame({
    'price': [float(t.price.as_double()) for t in ticks],
    'size': [float(t.size.as_double()) for t in ticks],
}, index=pd.to_datetime([t.ts_event for t in ticks], unit='ns'))
ohlc = tick_df['price'].resample('1min').ohlc().dropna()

fig, ax = plt.subplots(figsize=(14, 6))

# Plot price line (close)
ax.plot(ohlc.index, ohlc['close'], color='#333333', linewidth=1, alpha=0.7, label='ETH Price')

# Overlay trade entry/exit
for _, pos in positions_report.iterrows():
    entry_time = pd.Timestamp(pos['ts_opened'])
    exit_time = pd.Timestamp(pos['ts_closed'])
    entry_px = float(pos['avg_px_open'])
    exit_px = float(pos['avg_px_close'])
    pnl = pos['pnl']
    color = '#4CAF50' if pnl > 0 else '#F44336'
    marker = '^' if str(pos['entry']).strip() == 'BUY' else 'v'
    ax.scatter(entry_time, entry_px, marker=marker, color=color, s=80, zorder=5, edgecolors='black', linewidth=0.5)
    ax.scatter(exit_time, exit_px, marker='o', color=color, s=60, zorder=5, edgecolors='black', linewidth=0.5, alpha=0.7)
    ax.plot([entry_time, exit_time], [entry_px, exit_px], color=color, linewidth=1, alpha=0.5, linestyle='--')

ax.set_title('ETHUSDT Price with Trade Entries/Exits', fontsize=14, fontweight='bold')
ax.set_ylabel('Price (USDT)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.tight_layout()
plt.show()

In [ ]:
# === 4. PnL Distribution ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(positions_report['pnl'], bins=8, color='#9C27B0', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('PnL Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('PnL (USDT)')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3, axis='y')

# Box plot
bp = axes[1].boxplot(positions_report['pnl'], vert=True, patch_artist=True,
                     boxprops=dict(facecolor='#9C27B0', alpha=0.5),
                     medianprops=dict(color='black', linewidth=2))
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('PnL Box Plot', fontsize=13, fontweight='bold')
axes[1].set_ylabel('PnL (USDT)')
axes[1].grid(True, alpha=0.3, axis='y')

print(f'Mean PnL: {positions_report["pnl"].mean():+.4f} USDT')
print(f'Median PnL: {positions_report["pnl"].median():+.4f} USDT')
print(f'Std Dev: {positions_report["pnl"].std():+.4f} USDT')
print(f'Max Win: {positions_report["pnl"].max():+.4f} USDT')
print(f'Max Loss: {positions_report["pnl"].min():+.4f} USDT')
plt.tight_layout()
plt.show()

In [ ]:
engine.dispose()
print('Done.')